In [1]:
import json
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests

GAMMA_API = "https://gamma-api.polymarket.com"
DATA_API = "https://data-api.polymarket.com"
REQUEST_TIMEOUT = 20

OUTPUT_DIR = Path(".")


def get_json(url: str, params: Optional[Dict[str, Any]] = None) -> Any:
    response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def safe_json_loads(value: Any) -> Any:
    if not isinstance(value, str):
        return value
    try:
        return json.loads(value)
    except Exception:
        return value


def parse_float(value: Any) -> Optional[float]:
    try:
        if value is None:
            return None
        return float(value)
    except (TypeError, ValueError):
        return None


def to_float_or_zero(value: Any) -> float:
    parsed = parse_float(value)
    return parsed if parsed is not None else 0.0


def get_event_by_slug(slug: str) -> Dict[str, Any]:
    slug = slug.strip()
    if not slug:
        raise ValueError("Slug cannot be empty.")
    return get_json(f"{GAMMA_API}/events/slug/{slug}")


def get_event_by_id(event_id: str) -> Dict[str, Any]:
    event_id = str(event_id).strip()
    if not event_id:
        raise ValueError("Event ID cannot be empty.")
    return get_json(f"{GAMMA_API}/events/{event_id}")


def get_event_id_from_slug(slug: str) -> str:
    event = get_event_by_slug(slug)
    event_id = event.get("id")
    if event_id is None:
        raise ValueError(f"No event ID found for slug: {slug}")
    return str(event_id)


def find_markets_in_event(event: Dict[str, Any]) -> List[Dict[str, Any]]:
    for key in ("markets", "seriesMarkets", "children"):
        value = event.get(key)
        if isinstance(value, list):
            return value
    return []


def get_condition_id(market: Dict[str, Any]) -> Optional[str]:
    for key in ("conditionId", "condition_id"):
        value = market.get(key)
        if isinstance(value, str) and value:
            return value

    for key in ("condition",):
        value = market.get(key)
        if isinstance(value, str) and value:
            return value

    return None


def get_market_positions(
    condition_id: str,
    limit: int = 500,
    offset: int = 0,
    status: str = "ALL",
    sort_by: str = "TOTAL_PNL",
    sort_direction: str = "DESC",
) -> List[Dict[str, Any]]:
    """
    Fetch market positions from Polymarket.
    status='ALL' includes both open and closed/cashed-out positions.
    """
    return get_json(
        f"{DATA_API}/v1/market-positions",
        params={
            "market": condition_id,
            "status": status,
            "sortBy": sort_by,
            "sortDirection": sort_direction,
            "limit": limit,
            "offset": offset,
        },
    )


def normalize_position(position: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "proxyWallet": position.get("proxyWallet"),
        "name": position.get("name"),
        "asset": position.get("asset"),
        "size": position.get("size"),
        "avgPrice": position.get("avgPrice"),
        "currPrice": position.get("currPrice"),
        "currentValue": position.get("currentValue"),
        "cashPnl": position.get("cashPnl"),
        "totalBought": position.get("totalBought"),
        "realizedPnl": position.get("realizedPnl"),
        "totalPnl": position.get("totalPnl"),
        "outcome": position.get("outcome"),
        "outcomeIndex": position.get("outcomeIndex"),
        "verified": position.get("verified"),
        "profileImage": position.get("profileImage"),
        "conditionId": position.get("conditionId"),
    }


def rank_positions(
    positions: List[Dict[str, Any]],
    rank_by: str = "totalBought",
    top_n: int = 50,
) -> List[Dict[str, Any]]:
    ranked = sorted(
        positions,
        key=lambda x: to_float_or_zero(x.get(rank_by)),
        reverse=True,
    )
    return ranked[:top_n]


def split_yes_no(
    position_groups: List[Dict[str, Any]],
    rank_by: str = "totalBought",
    top_n: int = 50,
) -> Dict[str, List[Dict[str, Any]]]:
    yes_positions: List[Dict[str, Any]] = []
    no_positions: List[Dict[str, Any]] = []

    for group in position_groups:
        positions = group.get("positions", []) or []
        for pos in positions:
            normalized = normalize_position(pos)
            outcome = str(normalized.get("outcome", "")).strip().upper()

            if outcome == "YES":
                yes_positions.append(normalized)
            elif outcome == "NO":
                no_positions.append(normalized)

    return {
        "YES": rank_positions(yes_positions, rank_by=rank_by, top_n=top_n),
        "NO": rank_positions(no_positions, rank_by=rank_by, top_n=top_n),
    }


def extract_top_positions_for_event(
    event_id: str,
    rank_by: str = "totalBought",
    top_n: int = 50,
) -> Dict[str, Any]:
    event = get_event_by_id(event_id)
    event_title = event.get("title") or event.get("question") or f"Event {event_id}"
    event_slug = event.get("slug")
    markets = find_markets_in_event(event)

    if not markets:
        raise ValueError("No markets found inside the event response.")

    results: Dict[str, Any] = {
        "event_id": str(event_id),
        "event_slug": event_slug,
        "event_title": event_title,
        "rank_by": rank_by,
        "markets": [],
    }

    for idx, market in enumerate(markets, start=1):
        condition_id = get_condition_id(market)
        if not condition_id:
            continue

        question = market.get("question") or market.get("title") or f"Market {idx}"
        market_id = market.get("id")
        market_slug = market.get("slug")

        groups = get_market_positions(
            condition_id=condition_id,
            limit=500,
            offset=0,
            status="ALL",
            sort_by="TOTAL_PNL",  # server-side pre-sort; final ranking is local by rank_by
            sort_direction="DESC",
        )

        split = split_yes_no(groups, rank_by=rank_by, top_n=top_n)

        results["markets"].append(
            {
                "market_id": market_id,
                "market_slug": market_slug,
                "market_question": question,
                "condition_id": condition_id,
                "top_yes": split["YES"],
                "top_no": split["NO"],
            }
        )

    return results


def build_positions_dataframe(data: Dict[str, Any]) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for market in data.get("markets", []):
        market_question = market.get("market_question", "")
        market_id = market.get("market_id", "")
        market_slug = market.get("market_slug", "")
        condition_id = market.get("condition_id", "")

        for side_key, side_name in (("top_yes", "YES"), ("top_no", "NO")):
            holders = market.get(side_key, []) or []
            for rank, holder in enumerate(holders, start=1):
                rows.append(
                    {
                        "event_id": data.get("event_id", ""),
                        "event_slug": data.get("event_slug", ""),
                        "event_title": data.get("event_title", ""),
                        "market_id": market_id,
                        "market_slug": market_slug,
                        "market_question": market_question,
                        "condition_id": condition_id,
                        "side": side_name,
                        "rank": rank,
                        "name": holder.get("name") or "",
                        "proxyWallet": holder.get("proxyWallet") or "",
                        "verified": holder.get("verified"),
                        "asset": holder.get("asset") or "",
                        "size": to_float_or_zero(holder.get("size")),
                        "totalBought": to_float_or_zero(holder.get("totalBought")),
                        "avgPrice": to_float_or_zero(holder.get("avgPrice")),
                        "currPrice": to_float_or_zero(holder.get("currPrice")),
                        "currentValue": to_float_or_zero(holder.get("currentValue")),
                        "cashPnl": to_float_or_zero(holder.get("cashPnl")),
                        "realizedPnl": to_float_or_zero(holder.get("realizedPnl")),
                        "totalPnl": to_float_or_zero(holder.get("totalPnl")),
                        "outcomeIndex": holder.get("outcomeIndex"),
                    }
                )

    return pd.DataFrame(rows)


def save_highlighted_table_html(
    df: pd.DataFrame,
    output_file: Path,
    highlight_columns: Optional[List[str]] = None,
) -> None:
    if df.empty:
        output_file.write_text(
            "<html><body><h2>No data available</h2></body></html>",
            encoding="utf-8",
        )
        return

    if highlight_columns is None:
        highlight_columns = ["totalBought", "currentValue", "realizedPnl", "totalPnl"]

    display_df = df.copy()

    format_map = {
        "size": "{:,.2f}",
        "totalBought": "{:,.2f}",
        "avgPrice": "{:,.4f}",
        "currPrice": "{:,.4f}",
        "currentValue": "{:,.2f}",
        "cashPnl": "{:,.2f}",
        "realizedPnl": "{:,.2f}",
        "totalPnl": "{:,.2f}",
    }

    styler = display_df.style.format(format_map)

    for col in highlight_columns:
        if col in display_df.columns:
            styler = styler.background_gradient(cmap="Reds", subset=[col])

    styler = styler.set_properties(
        **{
            "text-align": "left",
            "border": "1px solid #d0d0d0",
            "font-size": "13px",
            "padding": "6px",
        }
    ).set_table_styles(
        [
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "100%"),
                    ("font-family", "Arial, sans-serif"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#f5f5f5"),
                    ("border", "1px solid #d0d0d0"),
                    ("padding", "6px"),
                    ("position", "sticky"),
                    ("top", "0"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #d0d0d0"),
                    ("padding", "6px"),
                ],
            },
        ]
    )

    html = f"""
    <html>
    <head>
        <meta charset="utf-8">
        <title>Top Historical Positions</title>
    </head>
    <body>
        <h2>Top Historical Positions by Event</h2>
        <p>Includes open and closed positions. Darker red indicates larger values.</p>
        {styler.to_html()}
    </body>
    </html>
    """

    output_file.write_text(html, encoding="utf-8")


def print_summary(data: Dict[str, Any]) -> None:
    print(f"Event title: {data.get('event_title')}")
    print(f"Event slug:  {data.get('event_slug')}")
    print(f"Event ID:    {data.get('event_id')}")
    print(f"Ranked by:   {data.get('rank_by')}")
    print("=" * 90)

    for market in data.get("markets", []):
        print(f"\nMarket: {market.get('market_question')}")
        print(f"Market ID: {market.get('market_id')}")
        print(f"Condition ID: {market.get('condition_id')}")
        print(f"Top YES rows: {len(market.get('top_yes', []))}")
        print(f"Top NO rows:  {len(market.get('top_no', []))}")


def main() -> None:
    slug = input("Enter Polymarket event slug: ").strip()

    event = get_event_by_slug(slug)
    event_id = str(event["id"])

    print("\nResolved slug to event:")
    print(f"Title: {event.get('title')}")
    print(f"Slug:  {event.get('slug')}")
    print(f"ID:    {event_id}")

    # rank_by can be changed to:
    # "totalBought"  -> biggest historical positions
    # "realizedPnl"  -> biggest realized winners
    # "totalPnl"     -> biggest total winners
    # "size"         -> biggest current/open-ish size, not what you want here
    rank_by = "totalBought"

    data = extract_top_positions_for_event(
        event_id=event_id,
        rank_by=rank_by,
        top_n=50,
    )
    df = build_positions_dataframe(data)

    json_file = OUTPUT_DIR / "top_positions_by_event.json"
    csv_file = OUTPUT_DIR / "top_positions_by_event.csv"
    html_file = OUTPUT_DIR / "top_positions_table.html"

    with json_file.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    df.to_csv(csv_file, index=False, encoding="utf-8")
    save_highlighted_table_html(df, html_file)

    print()
    print_summary(data)
    print("\nSaved files:")
    print(f"- {json_file}")
    print(f"- {csv_file}")
    print(f"- {html_file}")


if __name__ == "__main__":
    main()


Resolved slug to event:
Title: US strikes Iran by...?
Slug:  us-strikes-iran-by
ID:    114242

Event title: US strikes Iran by...?
Event slug:  us-strikes-iran-by
Event ID:    114242
Ranked by:   totalBought

Market: US strikes Iran by February 5, 2026?
Market ID: 1294628
Condition ID: 0x3bed62b0b7e3eb52c1f0d8a5d11edad1f74989038fc1cae2889cdbe96a248dfe
Top YES rows: 50
Top NO rows:  50

Market: US strikes Iran by January 31, 2026?
Market ID: 1092199
Condition ID: 0xabb86b080e9858dcb3f46954010e49b6f539c20036856c7f999395bfd58d01e6
Top YES rows: 50
Top NO rows:  50

Market: US strikes Iran by January 14, 2026?
Market ID: 1175504
Condition ID: 0x64b14a09a6cf9dc02b840bc83f4dcfd41ca6108544c47ecabc6e5d00bc15fd2e
Top YES rows: 50
Top NO rows:  50

Market: US strikes Iran by January 15, 2026?
Market ID: 1175524
Condition ID: 0x2d4d7f2eea43913d65c93351621a733f66ec079e111c278b34e8cc9e06ebe751
Top YES rows: 50
Top NO rows:  50

Market: US strikes Iran by January 18, 2026?
Market ID: 1175548
Condit